Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\24pasos_mlp_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 7)
Dimensiones de Y: (43765, 1)


In [9]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [10]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43765, 84)


Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 84)
Las dimensiones de testX son:  (8797, 84)
Las dimensiones de valX son:  (4333, 84)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

193/193 - 3s - 18ms/step - ia: 0.2241 - loss: 7.4205 - mae: 1.9212 - rmse: 2.5779 - smape: 1.4923 - val_ia: 0.2550 - val_loss: 0.5827 - val_mae: 0.5864 - val_rmse: 0.6873 - val_smape: 1.4143

Epoch 2/128                                           

193/193 - 0s - 2ms/step - ia: 0.2730 - loss: 2.6818 - mae: 1.2057 - rmse: 1.6075 - smape: 1.4534 - val_ia: 0.2565 - val_loss: 0.5400 - val_mae: 0.5706 - val_rmse: 0.6630 - val_smape: 1.5192

Epoch 3/128                                           

193/193 - 0s - 2ms/step - ia: 0.2947 - loss: 2.0120 - mae: 1.0422 - rmse: 1.3870 - smape: 1.4259 - val_ia: 0.2554 - val_loss: 0.5347 - val_mae: 0.5696 - val_rmse: 0.6588 - val_smape: 1.5803

Epoch 4/128                                           

193/193 - 0s - 2ms/step - ia: 0.2913 - loss: 1.6809 - mae: 0.9674 - rmse: 1.2733 - smape: 1.4415 - val_ia: 0.2550 - val_loss: 0.5296 - val_mae: 0.5669 - val_rmse: 0.6557 - val_smape: 1.5836

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



25/25 - 2s - 90ms/step - ia: 0.3439 - loss: 0.9952 - mae: 0.7355 - rmse: 0.9952 - smape: 1.3245 - val_ia: 0.2818 - val_loss: 0.4612 - val_mae: 0.4990 - val_rmse: 0.6678 - val_smape: 1.2510

Epoch 2/16                                                                      

25/25 - 0s - 6ms/step - ia: 0.4072 - loss: 0.8694 - mae: 0.6850 - rmse: 0.9329 - smape: 1.2046 - val_ia: 0.2896 - val_loss: 0.4689 - val_mae: 0.5141 - val_rmse: 0.6751 - val_smape: 1.3219

Epoch 3/16                                                                      

25/25 - 0s - 6ms/step - ia: 0.4460 - loss: 0.8186 - mae: 0.6612 - rmse: 0.8949 - smape: 1.1738 - val_ia: 0.3022 - val_loss: 0.4689 - val_mae: 0.4990 - val_rmse: 0.6724 - val_smape: 1.2240

Epoch 4/16                                                                      

25/25 - 0s - 6ms/step - ia: 0.4724 - loss: 0.7763 - mae: 0.6395 - rmse: 0.8914 - smape: 1.1418 - val_ia: 0.3045 - val_loss: 0.4797 - val_mae: 0.5074 - val_rmse: 0.6800 - val_smape: 1.227

...

770/770 - 3s - 4ms/step - ia: 0.3976 - loss: 1.0571 - mae: 0.7630 - rmse: 0.9668 - smape: 1.2514 - val_ia: 0.2159 - val_loss: 0.4966 - val_mae: 0.5248 - val_rmse: 0.5765 - val_smape: 1.2355

Epoch 2/8                                                                        

770/770 - 1s - 2ms/step - ia: 0.4181 - loss: 0.9139 - mae: 0.6971 - rmse: 0.8962 - smape: 1.2048 - val_ia: 0.2213 - val_loss: 0.4693 - val_mae: 0.5084 - val_rmse: 0.5563 - val_smape: 1.2102

Epoch 3/8                                                                        

770/770 - 1s - 2ms/step - ia: 0.4270 - loss: 0.8701 - mae: 0.6792 - rmse: 0.8753 - smape: 1.1874 - val_ia: 0.2276 - val_loss: 0.4672 - val_mae: 0.4960 - val_rmse: 0.5426 - val_smape: 1.1680

Epoch 4/8                                                                        

770/770 - 2s - 2ms/step - ia: 0.4331 - loss: 0.8454 - mae: 0.6688 - rmse: 0.8610 - smape: 1.1700 - val_ia: 0.2233 - val_loss: 0.4456 - val_mae: 0.4936 - val_rmse: 0.5373 - val_sm

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



49/49 - 2s - 46ms/step - ia: 0.3981 - loss: 1.1220 - mae: 0.7860 - rmse: 1.0460 - smape: 1.2689 - val_ia: 0.3060 - val_loss: 0.4948 - val_mae: 0.5392 - val_rmse: 0.6729 - val_smape: 1.2985

Epoch 2/128                                                                      

49/49 - 0s - 4ms/step - ia: 0.4136 - loss: 0.9373 - mae: 0.7167 - rmse: 0.9610 - smape: 1.2399 - val_ia: 0.3228 - val_loss: 0.4897 - val_mae: 0.5461 - val_rmse: 0.6784 - val_smape: 1.3207

Epoch 3/128                                                                      

49/49 - 0s - 4ms/step - ia: 0.4235 - loss: 0.9240 - mae: 0.7107 - rmse: 0.9533 - smape: 1.2251 - val_ia: 0.3118 - val_loss: 0.4978 - val_mae: 0.5447 - val_rmse: 0.6754 - val_smape: 1.3468

Epoch 4/128                                                                      

49/49 - 0s - 4ms/step - ia: 0.4291 - loss: 0.9028 - mae: 0.7010 - rmse: 0.9406 - smape: 1.2189 - val_ia: 0.3121 - val_loss: 0.4925 - val_mae: 0.5453 - val_rmse: 0.6738 - val_smape: 1.

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



25/25 - 4s - 173ms/step - ia: 0.2750 - loss: 1.0963 - mae: 0.7826 - rmse: 1.0286 - smape: 1.4421 - val_ia: 0.2621 - val_loss: 0.4640 - val_mae: 0.5199 - val_rmse: 0.6712 - val_smape: 1.3619

Epoch 2/128                                                                      

25/25 - 0s - 9ms/step - ia: 0.3522 - loss: 1.0109 - mae: 0.7534 - rmse: 0.9908 - smape: 1.3249 - val_ia: 0.3008 - val_loss: 0.4470 - val_mae: 0.4941 - val_rmse: 0.6605 - val_smape: 1.2163

Epoch 3/128                                                                      

25/25 - 0s - 9ms/step - ia: 0.3906 - loss: 0.9855 - mae: 0.7421 - rmse: 1.0236 - smape: 1.2767 - val_ia: 0.3226 - val_loss: 0.4540 - val_mae: 0.4911 - val_rmse: 0.6673 - val_smape: 1.1670

Epoch 4/128                                                                      

25/25 - 0s - 7ms/step - ia: 0.4080 - loss: 0.9576 - mae: 0.7354 - rmse: 0.9674 - smape: 1.2577 - val_ia: 0.3262 - val_loss: 0.4586 - val_mae: 0.4948 - val_rmse: 0.6713 - val_smape: 1

In [17]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.2, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.00016150678631378582, 'units': 2}


In [18]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.2, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.00016150678631378582, 'units': 2}
